# Korean Hate Speech Detection Kaggle Pipeline

이 노트북은 Kaggle 환경에서 바로 돌릴 수 있는 단일 파이프라인입니다.

- 기본 전략: `KcELECTRA` + 3-fold ensemble
- 옵션 전략: `K-MHaS binary pre-finetune -> Kaggle 3-class fine-tune`
- 출력: `submission.csv`, `submission_comments_label.csv`, `submission_comments_label_numeric.csv`

준비물:
- competition dataset를 Kaggle input으로 연결
- `K-MHaS` processed csv를 쓰려면 별도 Kaggle dataset으로 연결
- 인터넷이 꺼져 있으면 Hugging Face model download가 안 될 수 있으니 주의


In [ ]:
import gc
import json
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

COMP_DIR = Path('/kaggle/input/korean-hate-speech-detection')
KMHAS_DIR = Path('/kaggle/input/kmhas-processed')
WORK_DIR = Path('/kaggle/working')

MODEL_ID = 'beomi/KcELECTRA-base-v2022'
USE_KMHAS_PRETRAIN = True
USE_TITLE = False

FOLDS = 3
EPOCHS = 4
SEED = 42
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 1
USE_FP16 = False
USE_BF16 = False
USE_CLASS_WEIGHTS = True
LOG_EVERY = 100

LABELS = ['none', 'offensive', 'hate']
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    gib = 1024 ** 3
    print('gpu_free_gb:', round(free_bytes / gib, 2), 'gpu_total_gb:', round(total_bytes / gib, 2))
print({
    'MODEL_ID': MODEL_ID,
    'USE_KMHAS_PRETRAIN': USE_KMHAS_PRETRAIN,
    'USE_TITLE': USE_TITLE,
    'FOLDS': FOLDS,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE,
    'MAX_LENGTH': MAX_LENGTH,
    'GRAD_ACCUM': GRAD_ACCUM,
    'USE_FP16': USE_FP16,
    'USE_BF16': USE_BF16,
})


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_labeled_split(stem: str) -> pd.DataFrame:
    frame = pd.read_csv(COMP_DIR / f'{stem}.hate.csv')
    title_path = COMP_DIR / f'{stem}.news_title.txt'
    if title_path.exists():
        with title_path.open(encoding='utf-8') as handle:
            frame['news_title'] = [line.rstrip('\n') for line in handle]
    else:
        frame['news_title'] = ''
    frame['source_split'] = stem
    return frame


def load_competition_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = load_labeled_split('train')
    dev_df = load_labeled_split('dev')
    full_df = pd.concat([train_df, dev_df], ignore_index=True)
    full_df.insert(0, 'id', [f'SRC_{idx:05d}' for idx in range(len(full_df))])

    test_df = pd.read_csv(COMP_DIR / 'test.hate.no_label.csv')
    title_path = COMP_DIR / 'test.news_title.txt'
    if title_path.exists():
        with title_path.open(encoding='utf-8') as handle:
            test_df['news_title'] = [line.rstrip('\n') for line in handle]
    else:
        test_df['news_title'] = ''
    test_df.insert(0, 'id', [f'KG_{idx:05d}' for idx in range(len(test_df))])
    return full_df, test_df


set_seed(SEED)
train_df, test_df = load_competition_data()
print(train_df.shape, test_df.shape)
print(train_df['label'].value_counts().to_dict())


In [ ]:
class HateSpeechDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, use_title: bool = False) -> None:
        self.comments = frame['comments'].fillna('').tolist()
        self.titles = frame['news_title'].fillna('').tolist() if use_title else [''] * len(frame)
        self.labels = None
        if 'label' in frame.columns:
            self.labels = [LABEL_TO_ID[label] for label in frame['label'].tolist()]

    def __len__(self) -> int:
        return len(self.comments)

    def __getitem__(self, idx: int):
        item = {
            'comments': self.comments[idx],
            'news_title': self.titles[idx],
        }
        if self.labels is not None:
            item['label'] = self.labels[idx]
        return item


def make_collate_fn(tokenizer, max_length: int, use_title: bool = False):
    def collate(batch):
        comments = [row['comments'] for row in batch]
        titles = [row['news_title'] for row in batch] if use_title else [''] * len(batch)
        encoded = tokenizer(
            comments,
            titles,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        if 'label' in batch[0]:
            encoded['labels'] = torch.tensor([row['label'] for row in batch], dtype=torch.long)
        return encoded
    return collate


def move_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def score_labels(true_labels, pred_ids):
    pred_labels = [ID_TO_LABEL[idx] for idx in pred_ids]
    return {
        'weighted_f1': f1_score(true_labels, pred_labels, average='weighted'),
        'macro_f1': f1_score(true_labels, pred_labels, average='macro'),
        'accuracy': accuracy_score(true_labels, pred_labels),
    }


def build_model(model_name_or_path: str):
    return AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        ignore_mismatched_sizes=True,
    )


def evaluate(model, data_loader, device):
    model.eval()
    preds, trues = [], []
    use_autocast = device.type == 'cuda' and (USE_FP16 or USE_BF16)
    autocast_dtype = torch.float16 if USE_FP16 else torch.bfloat16
    with torch.no_grad():
        for batch in data_loader:
            batch = move_to_device(batch, device)
            labels = batch.pop('labels')
            autocast_ctx = torch.autocast(device_type=device.type, dtype=autocast_dtype) if use_autocast else nullcontext()
            with autocast_ctx:
                logits = model(**batch).logits
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            trues.extend(labels.cpu().tolist())
    return score_labels(pd.Series([ID_TO_LABEL[idx] for idx in trues]), np.array(preds))


def train_model(frame, eval_frame, tokenizer, model_name_or_path):
    collate_fn = make_collate_fn(tokenizer, MAX_LENGTH, use_title=USE_TITLE)
    train_loader = DataLoader(HateSpeechDataset(frame, use_title=USE_TITLE), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    eval_loader = DataLoader(HateSpeechDataset(eval_frame, use_title=USE_TITLE), batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = build_model(model_name_or_path).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16 and device.type == 'cuda')

    criterion = None
    if USE_CLASS_WEIGHTS:
        counts = frame['label'].value_counts()
        weights = torch.tensor([len(frame) / (len(LABELS) * counts[label]) for label in LABELS], dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)

    best_state = None
    best_metrics = None
    best_epoch = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader, start=1):
            batch = move_to_device(batch, device)
            labels = batch.pop('labels')
            use_autocast = device.type == 'cuda' and (USE_FP16 or USE_BF16)
            autocast_dtype = torch.float16 if USE_FP16 else torch.bfloat16
            autocast_ctx = torch.autocast(device_type=device.type, dtype=autocast_dtype) if use_autocast else nullcontext()
            with autocast_ctx:
                outputs = model(**batch)
                loss = nn.functional.cross_entropy(outputs.logits, labels) if criterion is None else criterion(outputs.logits, labels)
                loss = loss / GRAD_ACCUM

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item() * GRAD_ACCUM
            if step % LOG_EVERY == 0:
                print({'epoch': epoch, 'step': step, 'avg_loss': total_loss / step})

        metrics = evaluate(model, eval_loader, device)
        epoch_summary = {'epoch': epoch, 'train_loss': total_loss / len(train_loader), **metrics}
        history.append(epoch_summary)
        print(epoch_summary)
        if best_metrics is None or metrics['weighted_f1'] > best_metrics['weighted_f1']:
            best_metrics = metrics
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, best_metrics, best_epoch, history


def predict_proba(model, frame, tokenizer):
    collate_fn = make_collate_fn(tokenizer, MAX_LENGTH, use_title=USE_TITLE)
    loader = DataLoader(HateSpeechDataset(frame, use_title=USE_TITLE), batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    outputs = []
    model.eval()
    use_autocast = device.type == 'cuda' and (USE_FP16 or USE_BF16)
    autocast_dtype = torch.float16 if USE_FP16 else torch.bfloat16
    with torch.no_grad():
        for batch in loader:
            batch = move_to_device(batch, device)
            autocast_ctx = torch.autocast(device_type=device.type, dtype=autocast_dtype) if use_autocast else nullcontext()
            with autocast_ctx:
                logits = model(**batch).logits
            outputs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.vstack(outputs)


In [ ]:
def maybe_pretrain_on_kmhas(model_name_or_path: str) -> str:
    train_path = KMHAS_DIR / 'kmhas_train.csv'
    valid_path = KMHAS_DIR / 'kmhas_valid.csv'
    if not USE_KMHAS_PRETRAIN:
        print('skip KMHaS pretrain')
        return model_name_or_path
    if not train_path.exists() or not valid_path.exists():
        print('KMHaS files not found, skip pretrain:', KMHAS_DIR)
        return model_name_or_path

    kmhas_train = pd.read_csv(train_path)
    kmhas_valid = pd.read_csv(valid_path)
    bin_labels = ['not_hate', 'hate']
    bin_label_to_id = {label: idx for idx, label in enumerate(bin_labels)}
    bin_id_to_label = {idx: label for label, idx in bin_label_to_id.items()}

    class BinaryTextDataset(Dataset):
        def __init__(self, frame: pd.DataFrame) -> None:
            self.text = frame['text'].fillna('').tolist()
            self.labels = frame['hate_binary'].astype(int).tolist()
        def __len__(self):
            return len(self.text)
        def __getitem__(self, idx):
            return {'text': self.text[idx], 'label': self.labels[idx]}

    def make_binary_collate_fn(tokenizer, max_length: int):
        def collate(batch):
            encoded = tokenizer([row['text'] for row in batch], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            encoded['labels'] = torch.tensor([row['label'] for row in batch], dtype=torch.long)
            return encoded
        return collate

    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    collate_fn = make_binary_collate_fn(tokenizer, MAX_LENGTH)
    train_loader = DataLoader(BinaryTextDataset(kmhas_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(BinaryTextDataset(kmhas_valid), batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=2,
        id2label=bin_id_to_label,
        label2id=bin_label_to_id,
        ignore_mismatched_sizes=True,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16 and device.type == 'cuda')

    counts = kmhas_train['hate_binary'].value_counts()
    weights = torch.tensor([len(kmhas_train) / (2 * counts[idx]) for idx in [0, 1]], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    best_state = None
    best_f1 = -1.0
    optimizer.zero_grad()
    for epoch in range(1, 3):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader, start=1):
            batch = move_to_device(batch, device)
            labels = batch.pop('labels')
            use_autocast = device.type == 'cuda' and (USE_FP16 or USE_BF16)
            autocast_dtype = torch.float16 if USE_FP16 else torch.bfloat16
            autocast_ctx = torch.autocast(device_type=device.type, dtype=autocast_dtype) if use_autocast else nullcontext()
            with autocast_ctx:
                outputs = model(**batch)
                loss = criterion(outputs.logits, labels) / GRAD_ACCUM
            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
            total_loss += loss.item() * GRAD_ACCUM

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in valid_loader:
                batch = move_to_device(batch, device)
                labels = batch.pop('labels')
                logits = model(**batch).logits
                preds.extend(logits.argmax(dim=-1).cpu().tolist())
                trues.extend(labels.cpu().tolist())
        score = f1_score(trues, preds, average='weighted')
        print({'kmhas_epoch': epoch, 'train_loss': total_loss / len(train_loader), 'weighted_f1': score})
        if score > best_f1:
            best_f1 = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    out_dir = WORK_DIR / 'kcelectra_kmhas_binary'
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print('saved kmhas pretrained checkpoint to', out_dir)
    return str(out_dir)


model_path = maybe_pretrain_on_kmhas(MODEL_ID)
print('final model_path:', model_path)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_probs = np.zeros((len(train_df), len(LABELS)), dtype=np.float32)
test_probs = np.zeros((len(test_df), len(LABELS)), dtype=np.float32)
fold_metrics = []

for fold_idx, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['label']), start=1):
    print({'fold': fold_idx, 'status': 'start'})
    fold_train = train_df.iloc[train_idx].reset_index(drop=True)
    fold_valid = train_df.iloc[valid_idx].reset_index(drop=True)

    model, metrics, best_epoch, history = train_model(fold_train, fold_valid, tokenizer, model_path)
    oof_probs[valid_idx] = predict_proba(model, fold_valid, tokenizer)
    test_probs += predict_proba(model, test_df, tokenizer) / FOLDS
    fold_metrics.append({'fold': fold_idx, 'best_epoch': best_epoch, **metrics, 'history': history})
    print(fold_metrics[-1])

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

oof_pred_ids = oof_probs.argmax(axis=1)
oof_metrics = score_labels(train_df['label'], oof_pred_ids)
print('OOF metrics:', oof_metrics)


In [ ]:
test_pred_ids = test_probs.argmax(axis=1)
test_pred_labels = [ID_TO_LABEL[idx] for idx in test_pred_ids]

submission = pd.DataFrame({'label': test_pred_labels})
submission_comments = pd.DataFrame({'comments': test_df['comments'], 'label': test_pred_labels})
submission_comments_numeric = submission_comments.copy()
submission_comments_numeric['label'] = submission_comments_numeric['label'].map({'none': 0, 'offensive': 1, 'hate': 2})

metrics_payload = {
    'model_path': model_path,
    'use_kmhas_pretrain': USE_KMHAS_PRETRAIN,
    'use_title': USE_TITLE,
    'folds': FOLDS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'max_length': MAX_LENGTH,
    'grad_accum': GRAD_ACCUM,
    'use_fp16': USE_FP16,
    'use_bf16': USE_BF16,
    'oof_metrics': oof_metrics,
    'fold_metrics': fold_metrics,
}

submission.to_csv(WORK_DIR / 'submission.csv', index=False)
submission_comments.to_csv(WORK_DIR / 'submission_comments_label.csv', index=False)
submission_comments_numeric.to_csv(WORK_DIR / 'submission_comments_label_numeric.csv', index=False)
with (WORK_DIR / 'metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics_payload, handle, ensure_ascii=False, indent=2)

print('saved:', WORK_DIR / 'submission.csv')
print('saved:', WORK_DIR / 'submission_comments_label.csv')
print('saved:', WORK_DIR / 'submission_comments_label_numeric.csv')
print('saved:', WORK_DIR / 'metrics.json')
print(submission.head())
print(submission_comments_numeric['label'].value_counts().sort_index().to_dict())
